### **1. Recap: What are Runnables?**

*   **The Problem:** Early LangChain components (Prompts, LLMs, Output Parsers) weren't standardized. They each had different methods (e.g., `format`, `predict`, `parse`), making them difficult to connect for complex workflows.
*   **The Solution: Runnables.** LangChain introduced a standard interface. Now, all key components inherit from a base `Runnable` class and share a common set of methods, most importantly **`invoke`** . This standardization allows components to be easily chained together.

---

### **2. Two Types of Runnables**

LangChain Runnables can be divided into two main categories:

| Category | Description | Examples |
| :--- | :--- | :--- |
| **1. Task-Specific Runnables** | These are the core LangChain components that have been converted into Runnables. They have a specific purpose (e.g., creating prompts, calling an LLM). | `ChatPromptTemplate`, `ChatOpenAI`, `Retriever`, `OutputParser` |
| **2. Runnable Primitives** | These are fundamental building blocks that don't perform a specific task themselves. Instead, they are used to **orchestrate and structure the execution logic** of Task-Specific Runnables. They define *how* other runnables interact (sequentially, in parallel, conditionally, etc.). | `RunnableSequence`, `RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`, `RunnableBranch` |

This video focuses on **Runnable Primitives**.

---

#### **A. `RunnableSequence`**

*   **Purpose:** To execute multiple runnables **sequentially**. The output of the first runnable becomes the input for the next. It's the most fundamental way to create a pipeline.
*   **When to Use:** Whenever you have a linear chain of operations, like: Prompt -> LLM -> Output Parser.

### Initialize model and parser

In [1]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("Hugging_face_api_token")

# Create LLM endpoint
llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
)

# Wrap with chat interface
model = ChatHuggingFace(llm=llm)

In [4]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()

### define prompt templates 

In [ ]:
 # 1. Task-Specific Runnables
from langchain_core.prompts import PromptTemplate
generate_joke_prompt=PromptTemplate(
    template="Generate a joke about {topic}",
    input_variables=["topic"]
)
explain_joke_prompt=PromptTemplate(
    template="Explain the following joke: \n {joke}",
    input_variables=["joke"]
)

### define the chain and invoke

In [ ]:
from langchain_core.runnables import RunnableSequence

# 2. Create the Sequence
sequential_chain=RunnableSequence(generate_joke_prompt,model,parser,explain_joke_prompt,model,parser)

# Or use the more common pipe operator (LCEL):
# chain=generate_joke_prompt|model|parser|explain_joke_prompt|model|parser

result=sequential_chain.invoke({
    "topic":"love"
})

### Output

In [6]:
print(result)
print(type(result))

This joke plays on the double meaning of the phrase "make up," which can mean both "create or invent" and "fabricate or lie." The setup of the joke is that scientists don't trust atoms because they make up everything, referring to the fact that atoms are the fundamental building blocks of matter and thus contribute to the creation of everything around us.

The punchline then adds a playful twist by introducing the idea that atoms are also responsible for love, which is an abstract and intangible concept. The second part of the joke refers to a fictional movie title "The Atomic Force of Love," implying that love has a scientific explanation rooted in atoms.

The second part of the joke is a play on words involving the biochemist and his girlfriend. The biochemist calls his girlfriend a protein because of the amino acids that make up proteins, and the suffix "-ting" is added to make it sound like a pun on the word "tingling," suggesting a romantic connection. The overall joke is lighthea

### Graph visualization of chain

In [7]:
sequential_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       